# ಪಾಠ 01 - AI ಏಜೆಂಟ್‌ಗಳ ಪರಿಚಯ

**ಆರಂಭಕರಿಗಾಗಿ AI ಏಜೆಂಟ್‌ಗಳು** کورس್ನ ಮೊದಲ ಪಾಠಕ್ಕೆ ಸ್ವಾಗತ!

**AI ಏಜೆಂಟ್** ಎಂದರೆ ದೊಡ್ಡ ಭಾಷಾ ಮಾದರಿಯನ್ನು (LLM) ತನ್ನ ವಿಚಾರ ಗಣಕ ಯಂತ್ರವಾಗಿ ಉಪಯೋಗಿಸುವ ಮತ್ತು ಬಳಕೆದಾರರ ಪರವಾಗಿ ಗುರಿಯನ್ನು ಸಾಧಿಸಲು ನೈಜ ಲೋಕದಲ್ಲಿ *ಕ್ರಿಯೆಗಳನ್ನು* ಕೈಗೊಳ್ಳುವುದು — APIಗಳನ್ನು ಕರೆಸುವುದು, ಡೇಟಾಬೇಸನ್ನು ಪ್ರಶ್ನಿಸುವುದು, ಅಥವಾ ಕೋಡ್ ಚಲಿಸುವುದು ಎಂಬುದನ್ನು ಮಾಡಬಲ್ಲ ಪೋಗ್ರಾಂ.

ಈ ನೋಟ್‌ಬುಕ್ ನಲ್ಲಿ ನೀವು ನಿಮ್ಮ ಮೊದಲ ಏಜೆಂಟ್ ಅನ್ನು ನಿರ್ಮಿಸಲಿದ್ದೀರಿ: **ಪ್ರಯಾಣ ಏಜೆಂಟ್** ಇದು ರಜಾ ತಾಣಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡುತ್ತದೆ. ಈ ಮಾರ್ಗದಲ್ಲಿ ನೀವು ಹೇಗೆ ಕಲಿಯುತ್ತೀರಿ:

1. **Microsoft Agent Framework** ಉಪಯೋಗಿಸಿ Microsoft Foundry Agent Service ಗೆ ಸಂಪರ್ಕ ಸಾಧಿಸುವುದು.
2. ಏಜೆಂಟ್ ಗೆ ಒಂದು **ಉಪಕರಣ** ಒದಗಿಸುವುದು — ಇದು ಪ್ರತ್ಯಕ್ಷ ಪೈಥಾನ್ ಫಂಕ್ಷನ್ ಆಗಿದ್ದು ಸಾಧ್ಯವಿರುವ ಕರೆಸಬಲ್ಲದು.
3. ಏಜೆಂಟ್ ಅನ್ನು ಚಲಾಯಿಸಿ ಅದರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪರಿಶೀಲಿಸುವುದು.
4. token token ಆಗಿ ಏಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆ ಸ್ಟ್ರೀಮ್ ಮಾಡುವುದು.


## ಸೆಟಪ್

ಈ ನೋಟುಬುಕ್ ಅನ್ನು ರನ್ ಮಾಡುವ ಮೊದಲು, ನೀವು ಕೆಳಗಿನವನ್ನೂ ಖಚಿತಪಡಿಸಿಕೊಳ್ಳಿ:

1. **ಡಿಪ್ಲಾಯ್ ಮಾಡಲಾದ ಚಾಟ್ ಮಾದರಿಯೊಂದಿಗೆ Microsoft Foundry ಪ್ರಾಜೆಕ್ಟ್** (ಉದಾಹರಣೆಗಾಗಿ `gpt-4.1-mini`).
2. **Azure CLI ಸಹಿತ ಲಾಗಿನ್ ಆಗಿರುವುದು** — ನಿಮ್ಮ ಟರ್ಮಿನಲ್‌ನಲ್ಲಿ `az login` ಅನ್ನು ರನ್ ಮಾಡಿ.
3. **ಅವಶ್ಯಕವಾದ ಪರಿಸರ ಚರಗಳನ್ನು ಸೆಟ್ ಮಾಡಿ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — ನಿಮ್ಮ Microsoft Foundry ಪ್ರಾಜೆಕ್ಟ್ ಎಂಡ್‌ಪಾಯಿಂಟ್.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ನಿಮ್ಮ ಡಿಪ್ಲಾಯ್ ಮಾಡಲಾದ ಮಾದರಿಯ ಹೆಸರು.

ಕೆಳಗಿನ ಸೆಲ್ ನಿಮಗೆ ಬೇಕಾದ Python ಪ್ಯಾಕೇಜ್‌ಗಳನ್ನು ಇನ್‌ಸ್ಟಾಲ್ ಮಾಡುತ್ತದೆ.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## ನಿಮ್ಮ ಮೊದಲ ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸುವುದು

ಏಜೆಂಟ್‌ಗೆ ಎರಡು ವಿಷಯಗಳು ಬೇಕಾಗಿವೆ:

- **ನಿರ್ದೇಶನಗಳು** ಅದು *ಯಾರು* ಮತ್ತು *ಹೀಗೆ ನಡೆದುಕೊಳ್ಳಬೇಕು* (ಒಂದು ವ್ಯವಸ್ಥೆ ಸೂಚನೆ) ಎಂದು ತಿಳಿಸುತ್ತವೆ.
- **ಸಾಧನೆಗಳು** — `@tool` ಅಲಂಕೃತ Python ಫಂಕ್ಷನ್‌ಗಳು, ಏಜೆಂಟ್ ಮಾಹಿತಿ ಪಡೆಯಲು ಅಥವಾ ಕ್ರಿಯೆಗಳು ಮಾಡಲು ಬಳಸಿಕೊಳ್ಳಬಹುದು.

ಕೆಳಗೆ ನಾವು ಜನಪ್ರಿಯ ರಜಾ ಸ್ಥಳಗಳ ಪಟ್ಟಿ ನೀಡುವ ಒಂದು ಸರಳ ಸಾಧನವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತೇವೆ. ಬಳಕೆದಾರರು ಪ್ರವಾಸಿ ಸಲಹೆಗಳನ್ನು ಕೇಳಿದಾಗ ಏಜೆಂಟ್ ಈ ಸಾಧನವನ್ನು ಬಳಸುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ಸ್ಟ್ರೀಮಿಂಗ್ ಪ್ರತಿಕ್ರಿಯೆಗಳು

ಹೆಚ್ಚು ಪರಸ್ಪರ ಕ್ರಿಯಾಶೀಲ ಅನುಭವಕ್ಕಾಗಿ ನೀವು ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು **ಸ್ಟ್ರೀಮ್** ಮಾಡಬಹುದು. ಪೂರ್ಣ ಉತ್ತರಕ್ಕಾಗಿ ಕಾಯುವುದರಿಂದ ಬದಾಗಿ, ಏಜೆಂಟ್ ನಿರ್ಮಿಸಿರುವಂತೆ ಪಠ್ಯದ ತುಂಡುಗಳನ್ನು ನೀಡುತ್ತದೆ. ನೀವು ಫಲಿತಾಂಶವನ್ನು ನೈಜ ಸಮಯದಲ್ಲಿ ಪ್ರದರ್ಶಿಸಲು ಬಯಸುವ ಚಾಟ್ ಇಂಟರ್ಫೇಸುಗಳಲ್ಲಿ ಇದು ವಿಶೇಷವಾಗಿ ಉಪಯುಕ್ತವಾಗಿದೆ.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಹೇಗೆ ಕಲಿತಿರಿ:

- **Provider ಅನ್ನು ರಚಿಸುವುದು** Microsoft Foundry Agent Service ಗೆ `FoundryChatClient` ಮೂಲಕ ಸಂಪರ್ಕಸ್ಥಗೊಳಿಸುವುದು.
- **ಟೂಲ್ ಅನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವುದು** `@tool` ಡೇಕೊರೇಟರ್ ಬಳಸಿ, ಅರ್ಥಾತ್ ಏಜಂಟ್ ನಿಮ್ಮ Python ಕಾರ್ಯಗಳನ್ನು ಕರೆಸಲು ಸಾಧ್ಯವಾಗುವುದು.
- **ಏಜಂಟ್ ಅನ್ನು ಚಾಲನೆ ಮಾಡುವುದು** ಬಳಕೆದಾರ ಸಂದೇಶದೊಂದಿಗೆ ಮತ್ತು ಅದರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಮುದ್ರಿಸುವುದು.
- **ಪ್ರತಿಕ್ರಿಯೆಗಳาสต್ರೀಮ್ ಮಾಡುವುದು** ನೈಜ-ಸಮಯ ಫಲಿತಾಂಶಕ್ಕಾಗಿ.

ಮುಂದಿನ ಪಾಠದಲ್ಲಿ, ನಾವು ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್‌ಗಳನ್ನು ಹೆಚ್ಚು ಆಳವಾಗಿ ಪರಿಶೀಲಿಸಿ, ಏಜೆಂಟ್‌ಗಳಿಗೆ ಶಕ್ತಿಶಾಲಿ ಟೂಲ್ಸ್‌ ಮತ್ತು ಬಹು-ಹಂತ ವಿವೇಚನೆ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ನೀಡುವ ಕಲಿಕೆಯನ್ನು ಮಾಡುತ್ತೇವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
